# StockGen Upscaler — Google Colab Test
ทดสอบ Real-ESRGAN + GFPGAN ก่อน deploy ไป RunPod

In [ ]:
# 1) Install dependencies
import sys, os, types
!pip install -q realesrgan gfpgan opencv-python-headless matplotlib pillow

import torchvision.transforms.functional as F_tv
shim = types.ModuleType('torchvision.transforms.functional_tensor')
shim.rgb_to_grayscale = F_tv.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = shim
print('Dependencies installed')

In [ ]:
# 2) Load models with state dict conversion (supports all models including UltraSharp)
import torch
import urllib.request
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from gfpgan import GFPGANer
import cv2, base64, tempfile, numpy as np

MODELS = {
    'x2plus': {'url': 'https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth',
               'file': 'RealESRGAN_x2plus.pth', 'scale': 2, 'block': 23},
    'x4plus': {'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
               'file': 'RealESRGAN_x4plus.pth', 'scale': 4, 'block': 23},
    'anime': {'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
              'file': 'RealESRGAN_x4plus_anime_6B.pth', 'scale': 4, 'block': 6},
    'ultrasharp': {'url': 'https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth',
                   'file': '4x-UltraSharp.pth', 'scale': 4, 'block': 23},
}

weights_dir = './weights'
os.makedirs(weights_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def convert_state_dict(state_dict, num_block):
    if not any(k.startswith('model.0.') for k in state_dict.keys()):
        return state_dict
    print('Converting classic ESRGAN keys to RRDBNet...')
    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith('model.0.'):
            new_sd[k.replace('model.0.', 'conv_first.')] = v
        elif k.startswith('model.1.sub.'):
            parts = k.split('.')
            try:
                block_idx = int(parts[3])
                if block_idx == num_block:
                    new_sd[k.replace(f'model.1.sub.{block_idx}.', 'conv_body.')] = v
                else:
                    nk = k.replace('model.1.sub.', 'body.')
                    nk = nk.replace('.RDB', '.rdb')
                    for i in range(1, 6):
                        nk = nk.replace(f'.conv{i}.0.', f'.conv{i}.')
                    new_sd[nk] = v
            except:
                new_sd[k] = v
        elif k.startswith('model.3.'):
            new_sd[k.replace('model.3.', 'conv_up1.')] = v
        elif k.startswith('model.6.'):
            new_sd[k.replace('model.6.', 'conv_up2.')] = v
        elif k.startswith('model.8.'):
            new_sd[k.replace('model.8.', 'conv_hr.')] = v
        elif k.startswith('model.10.'):
            new_sd[k.replace('model.10.', 'conv_last.')] = v
        else:
            new_sd[k] = v
    return new_sd

def get_upsampler(model_key):
    cfg = MODELS[model_key]
    model_path = os.path.join(weights_dir, cfg['file'])
    if not os.path.exists(model_path):
        print(f'Downloading {cfg["file"]}...')
        urllib.request.urlretrieve(cfg['url'], model_path)
    try:
        loadnet = torch.load(model_path, map_location='cpu')
        if 'params' in loadnet:
            state_dict = loadnet['params']
        elif 'params_ema' in loadnet:
            state_dict = loadnet['params_ema']
        else:
            state_dict = loadnet
        converted = convert_state_dict(state_dict, cfg['block'])
        torch.save({'params': converted}, model_path)
    except Exception as e:
        print(f'Note: {e}')
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=cfg['block'], num_grow_ch=32, scale=cfg['scale'])
    return RealESRGANer(scale=cfg['scale'], model_path=model_path,
                        model=model, tile=800, tile_pad=10, pre_pad=10,
                        half=device.type == 'cuda', device=device), cfg['scale']

def get_face_enhancer(upsampler, scale):
    path = os.path.join(weights_dir, 'GFPGANv1.3.pth')
    if not os.path.exists(path):
        print('Downloading GFPGANv1.3.pth (~340MB)...')
        urllib.request.urlretrieve(
            'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth', path)
    return GFPGANer(model_path=path, upscale=scale, arch='clean',
                    channel_multiplier=2, bg_upsampler=upsampler)

print('Functions ready')

In [ ]:
# 3) Run test
import matplotlib.pyplot as plt
from PIL import Image
import io

TEST_IMAGE = 'https://raw.githubusercontent.com/TencentARC/GFPGAN/master/inputs/whole_imgs/00.jpg'
MODEL = 'x2plus'  # x2plus, x4plus, ultrasharp, anime
FACE_ENHANCE = True

upsampler, scale = get_upsampler(MODEL)

tmp_dir = tempfile.mkdtemp()
in_path = os.path.join(tmp_dir, 'in.jpg')
urllib.request.urlretrieve(TEST_IMAGE, in_path)
img = cv2.imread(in_path, cv2.IMREAD_COLOR)

if FACE_ENHANCE:
    face_enhancer = get_face_enhancer(upsampler, scale)
    _, _, output = face_enhancer.enhance(img, has_aligned=False,
                                         only_center_face=False, paste_back=True)
else:
    output, _ = upsampler.enhance(img, outscale=scale)

out_path = os.path.join(tmp_dir, 'out.png')
cv2.imwrite(out_path, output)

img_orig = Image.open(in_path)
img_out = Image.open(out_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
axes[0].imshow(img_orig)
axes[0].set_title(f'Before\n{img_orig.size[0]}x{img_orig.size[1]}', fontsize=12)
axes[0].axis('off')
axes[1].imshow(img_out)
axes[1].set_title(f'After x{scale}\n{img_out.size[0]}x{img_out.size[1]}', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print('Done!')